# Day 3 · 실습 ③ — ResNet18 Feature Extraction과 Fine-tuning 비교

**연계 이론:** Day 3 이론 1(Pre-training과 Transfer Learning), 이론 2(전이학습의 학습 전략)  
**목표:** 사전학습 ResNet18에서 `Backbone 동결`과 `layer4 동결 해제`를 비교합니다.

### 실습에서 확인할 것
1. 사전학습 ResNet18의 Backbone과 Classification Head 구분
2. `requires_grad`에 따른 학습 가능 파라미터 수 비교
3. 같은 데이터·분할·초기 Head·학습률·학습 횟수에서 두 조건 학습
4. Training/Validation Loss 곡선과 데이터 증강 결과 시각화

**50분 운영 예시:** 데이터/환경 5분 → 핵심 코드 10분 → 두 조건 실행 20분 → 결과 비교 10분 → 정리 5분.

> 두 조건의 결과를 **원인 하나로 단정하지 말고** 먼저 동결 범위와 학습 파라미터 차이를 관찰합니다. 실행 목표 시간은 사전 환경 준비 및 모델 다운로드를 제외한 교육용 목표입니다.

## 0. 실행 전 확인

- 실습 환경: Python, PyTorch, TorchVision, NumPy, Pandas, Matplotlib, Pillow
- GPU: 가능하면 **RTX 4090** 사용. GPU가 없으면 CPU로 실행되지만 학습 시간이 더 걸립니다.
- 최초 실행 시 **ResNet18 사전학습 가중치**와 PyTorch 공식 **개미·벌(Hymenoptera) 데이터셋**을 다운로드합니다.
- PyTorch/CUDA는 교육기관에서 사전 설치한 버전을 유지하세요. [공식 전이학습 튜토리얼](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- 저장 경로를 별도로 지정하려면 `DAY3_DATA_ROOT` 환경변수로 데이터 폴더를 설정할 수 있습니다.

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. 데이터 준비와 학습 조건

PyTorch 튜토리얼의 `ants` / `bees` 데이터셋을 사용합니다. 각 실험에 **같은 Subset**을 적용하고, 실험 비교에서는 **결정적인(고정) 입력 전처리**를 사용합니다. 무작위 증강은 별도 셀에서 눈으로 확인합니다. 학습 조건을 하나씩 비교하기 위한 설계입니다.

> 샘플 수·Epoch·Batch Size는 실행 시간 목표에 맞춘 초기값입니다. 실제 RTX 4090 동시 실습 시간은 운영 환경에서 점검해야 합니다.

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(Path.cwd() / "data" / "hymenoptera_data"))).expanduser().resolve()
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / "train" / "ants", root / "train" / "bees",
                root / "val" / "ants", root / "val" / "bees"]
    if all(p.is_dir() for p in required):
        print(f"데이터셋 준비됨: {root}")
        return root

    # PyTorch 공식 튜토리얼의 공개 개미/벌 데이터셋을 1회 내려받습니다.
    # 공동 실습에서는 강사가 미리 다운로드하여 DAY3_DATA_ROOT를 지정하세요.
    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.is_file():
        print("공식 PyTorch 개미/벌 데이터셋 다운로드 중...")
        tmp_file = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, tmp_file.open("wb") as out:
                shutil.copyfileobj(response, out)
            tmp_file.replace(archive)
        finally:
            tmp_file.unlink(missing_ok=True)

    with zipfile.ZipFile(archive) as zf:
        target_base = root.parent.resolve()
        for entry in zf.infolist():
            dest = (target_base / entry.filename).resolve()
            if not dest.is_relative_to(target_base):
                raise ValueError(f"압축 경로가 허용 범위를 벗어났습니다: {entry.filename}")
        zf.extractall(target_base)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"데이터셋의 train/val/ants/bees 폴더를 찾지 못했습니다: {root}")
    print(f"데이터셋 다운로드 완료: {root}")
    return root

ensure_dataset()

In [ ]:
# [학생이 바꿀 수 있는 설정] 실행 시간이 길면 EPOCHS를 2로 낮춰 관찰합니다.
TRAIN_PER_CLASS = 32
VAL_PER_CLASS = 12
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 3e-4

# 무작위 증강은 아래 별도 관찰 셀에서만 사용합니다.
base_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
full_train = datasets.ImageFolder(DATA_ROOT / "train", transform=base_transform)
full_val = datasets.ImageFolder(DATA_ROOT / "val", transform=base_transform)
assert full_train.classes == full_val.classes == ["ants", "bees"]
CLASS_NAMES = full_train.classes

def stratified_indices(dataset, per_class, seed):
    rng = np.random.default_rng(seed)
    indices = []
    for label in range(len(dataset.classes)):
        matched = np.flatnonzero(np.asarray(dataset.targets) == label)
        if len(matched) < per_class:
            raise ValueError(f"클래스 {label}: {per_class}개보다 적은 이미지")
        indices.extend(rng.choice(matched, size=per_class, replace=False).tolist())
    return sorted(indices)

train_set = Subset(full_train, stratified_indices(full_train, TRAIN_PER_CLASS, SEED))
val_set = Subset(full_val, stratified_indices(full_val, VAL_PER_CLASS, SEED))
print("classes:", CLASS_NAMES)
print("train:", len(train_set), "val:", len(val_set))
print("첫 텐서 Shape:", tuple(train_set[0][0].shape))

## 2. 같은 이미지에 데이터 증강 적용하기

이론 2의 Flip, Crop, Rotation, Color Jitter를 관찰합니다. 여기서는 증강 결과만 비교하고 **학습에 입력하는 데이터는 그대로 유지**합니다. 따라서 두 학습 전략의 비교에서 증강이 별도의 실험 변수가 되지 않습니다.

In [ ]:
example_path, _ = full_train.samples[0]
with Image.open(example_path) as im:
    original = im.convert("RGB")
augments = {
    "Original": lambda img: img,
    "Horizontal Flip": transforms.RandomHorizontalFlip(p=1.0),
    "Random Crop": transforms.RandomResizedCrop((224, 224), scale=(0.65, 0.85)),
    "Rotation": transforms.RandomRotation((25, 25)),
    "Color Jitter": transforms.ColorJitter(brightness=0.65, contrast=0.5),
}
fig, axes = plt.subplots(1, len(augments), figsize=(15, 3.5))
for ax, (name, operation) in zip(axes, augments.items()):
    ax.imshow(operation(original.copy()))
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("Same image, different augmentations")
plt.tight_layout()
plt.show()

## 3. ResNet18 분류 Head 교체와 동결 전략

같은 `ResNet18_Weights.IMAGENET1K_V1`을 불러오며 새 Head는 항상 같은 Random Seed로 초기화합니다. 두 조건 모두 학습 가능한 Head는 `Linear(512, 2)`입니다.

| 조건 | Backbone | 새로운 Head |
|---|---|---|
| Feature Extraction | 전부 동결 | 학습 |
| Partial Fine-tuning | `layer4` 동결 해제 | 학습 |

**추가 통제:** 소규모 데이터 비교에서 BatchNorm의 Running Mean/Variance가 변수가 되지 않도록 두 실험 모두 해당 통계는 고정합니다(`BatchNorm.eval()`). `layer4`의 학습 가능한 가중치 자체는 계속 업데이트됩니다.

In [ ]:
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1

def create_model(strategy):
    if strategy not in ("feature_extraction", "partial_finetuning"):
        raise ValueError("올바르지 않은 학습 전략")
    torch.manual_seed(SEED)
    model = models.resnet18(weights=WEIGHTS)
    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    if strategy == "partial_finetuning":
        for param in model.layer4.parameters():
            param.requires_grad = True
    return model.to(DEVICE)

def trainable_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for strategy in ("feature_extraction", "partial_finetuning"):
    model = create_model(strategy)
    names = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"{strategy:22} | 학습 가능: {trainable_count(model):>10,} / 전체: "
          f"{sum(p.numel() for p in model.parameters()):,}")
    print("학습 대상 계층 예:", names[:5], "...", names[-2:])
    del model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 4. 두 조건 학습하고 Loss 비교하기

- 동일한 훈련/검증 이미지와 변환
- 동일한 사전학습 가중치와 새 Head의 초기값
- 동일한 Epoch 수, Optimizer(`AdamW`), Learning Rate
- 차이로 설정하는 항목: **동결 해제 범위**

아래 코드는 학생이 처음부터 작성하는 활동이 아닙니다. 학습 파이프라인의 핵심 설정을 확인한 뒤 실행합니다.

In [ ]:
def make_loaders():
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                            generator=generator, num_workers=0,
                            pin_memory=(DEVICE.type == "cuda")),
        "val": DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda")),
    }

def one_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    # 학습 가능한 affine 파라미터와 무관하게 Running Mean/Var만 고정합니다.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_seen += images.size(0)
    return total_loss / total_seen, total_correct / total_seen

def run_experiment(strategy):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    loaders = make_loaders()
    model = create_model(strategy)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE
    )
    history = []
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = one_epoch(model, loaders["train"], criterion, optimizer)
        val_loss, val_acc = one_epoch(model, loaders["val"], criterion)
        history.append({"strategy": strategy, "epoch": epoch,
                        "train_loss": train_loss, "val_loss": val_loss,
                        "train_acc": train_acc, "val_acc": val_acc})
        print(f"{strategy} | Epoch {epoch}/{EPOCHS} | "
              f"train_loss {train_loss:.4f} / val_loss {val_loss:.4f} | "
              f"val_acc {val_acc:.3f}")
    result = {
        "strategy": strategy,
        "trainable_params": trainable_count(model),
        "elapsed_sec": time.perf_counter() - start,
        "history": history,
    }
    del model, optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result

results = [run_experiment("feature_extraction"),
           run_experiment("partial_finetuning")]

## 5. 결과 표와 그래프 확인

**관찰:** 두 전략의 학습 가능 파라미터 수가 실제로 다른가요? Loss 곡선의 변화가 어떻게 다른가요? 검증 Loss가 훈련 Loss와 같은 방향으로 움직이나요? 소규모·짧은 학습 결과로 어느 한 전략이 항상 더 좋다고 일반화하지 않습니다.

In [ ]:
summary = pd.DataFrame([{
    "strategy": r["strategy"],
    "trainable_params": r["trainable_params"],
    "elapsed_sec": round(r["elapsed_sec"], 2),
    "final_train_loss": round(r["history"][-1]["train_loss"], 4),
    "final_val_loss": round(r["history"][-1]["val_loss"], 4),
    "final_val_acc": round(r["history"][-1]["val_acc"], 3),
} for r in results])
display(summary)

history_df = pd.DataFrame([row for r in results for row in r["history"]])
fig, ax = plt.subplots(figsize=(8, 4.5))
for strategy in history_df["strategy"].unique():
    frame = history_df[history_df["strategy"] == strategy]
    ax.plot(frame["epoch"], frame["train_loss"], marker="o",
            label=f"{strategy} / train")
    ax.plot(frame["epoch"], frame["val_loss"], marker="s", linestyle="--",
            label=f"{strategy} / val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Feature Extraction vs Partial Fine-tuning")
ax.set_xticks(list(range(1, EPOCHS + 1)))
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

result_dir = Path.cwd() / "results"
result_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(result_dir / "day3_02_strategy_summary.csv", index=False, encoding="utf-8-sig")
history_df.to_csv(result_dir / "day3_02_loss_history.csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", result_dir)

## 6. 결과 해석과 핵심 질문

1. Feature Extraction에서 왜 `fc`의 파라미터만 학습되나요?
2. `layer4`를 동결 해제하면 학습 가능한 파라미터 수는 어떻게 달라지나요?
3. **두 실험에서 고정한 조건**과 **변경한 조건**은 각각 무엇인가요?
4. Training Loss와 Validation Loss를 함께 보는 이유는 무엇인가요?
5. Flip·Crop·Color Jitter의 효과를 비교하려면 다음 실험을 어떻게 분리해야 할까요?

**다음 학습 연결:** 새로운 분류 Head를 학습하지 않고, 사전학습 모델의 특징 벡터 자체를 이용해 이미지들을 비교하는 방법을 DINOv3 실습에서 살펴봅니다.